# Agentomics - a PydanticAI-powered autonomous ML Agent

In [ ]:
%%capture
!pip install -q condacolab pydantic-ai
import condacolab
condacolab.install()

In [ ]:
!git clone https://github.com/BioGeMT/agentomics-ml.git
!./agentomics-ml/scripts/download_example_dataset.sh --dataset "AGO2_CLASH_Hejret2023"

In [ ]:
from pathlib import Path
openrouter_api_key = ""  # Paste the workshop key here, for example: sk-or-v1-...

env_path = Path("agentomics-ml") / ".env"
env_path.write_text(f"OPENROUTER_API_KEY={openrouter_api_key}\n")

## Run Agentomics

In [ ]:
!./agentomics-ml/run.sh --local --dataset AGO2_CLASH_Hejret2023 --model openai/gpt-5.4-mini --iterations 1 --val-metric AUROC --cpu-only

# Pydantic-AI connection

In [ ]:
# Example of Structured output used in Agentomics
from pydantic import BaseModel, Field
from pydantic_ai import ModelRetry
import os

class ModelTrainingOutput(BaseModel):
    path_to_train_file: str = Field(description="Absolute path to the generated 'train.py'")
    path_to_model_file: str = Field(description="Absolute path to the trained model file")
    path_to_artifacts_dir: str = Field(
        description=(
            "Absolute path to the folder with artifacts produced by training. "
            "Must be called 'training_artifacts'. "
            "(This folder should be the parent of path_to_model_file and a sibling to train.py)"
        )
    )
    training_summary: str = Field(
        description="Short summary of the training implementation. Don't include any metrics in this summary."
    )
    unresolved_issues: str | None = Field(
        description=(
            "Issues that remain unresolved and could impact performance and/or metrics. "
            "(e.g. expected GPU to be available but is inaccessible during training, "
            "foundation model could not be loaded, etc...). Can be empty."
        )
    )
    
    

In [ ]:
# How we validate the structured output for model training?

@model_training_agent.output_validator
def validate_model_training_output(result: dict) -> ModelTrainingOutput:
    if not os.path.exists(result.path_to_train_file):
        raise ModelRetry(f"Train file does not exist. {result.path_to_train_file}")
    if Path(result.path_to_train_file).name.strip() != "train.py":
        raise ModelRetry(f"Train file must be called 'train.py' , currently is named {Path(result.path_to_train_file).name.strip()}")
    if not os.path.exists(result.path_to_model_file):
        raise ModelRetry(f"Model file does not exist at {result.path_to_model_file}")
    try:
        script_runs_and_produces_artifacts(result.path_to_train_file)
    except Exception as e:
        raise ModelRetry(f"Train script {result.path_to_train_file} does not run or produce model file {result.path_to_model_file}") from e
    return result